# 02 - Vardiya Personel Optimizasyonu: Simülasyon + Bayes Optimizasyonu

Bu notebook, endüstri mühendisliği için simülasyon tabanlı optimizasyon örneğidir.

Problem:

- Sistem üç vardiyadan oluşuyor.
- Her vardiyada farklı müşteri geliş yoğunluğu var.
- Her vardiyada kaç personel çalışacağını seçiyoruz.
- Daha çok personel beklemeyi azaltıyor fakat işgücü maliyetini artırıyor.
- Amaç, toplam ekonomik maliyeti düşürmek.

Bu örnekte karar değişkenleri tamsayıdır.

## 1. Önceki basit örneğin neden `10, 10, 10` bulduğu

Yalnızca bekleme süresi minimize edilirse, personel sayısını artırmanın herhangi bir maliyeti yoktur.

Dolayısıyla:

```text
daha fazla personel -> daha az bekleme
```

olduğu sürece optimizer'ın tüm karar değişkenlerini üst sınıra götürmesi beklenir.

Bu nedenle gerçekçi amaç fonksiyonuna personel maliyeti eklenmelidir.

## 2. Amaç fonksiyonunda birim tutarlılığı

Şu ifade sorunludur:

```text
toplam bekleme dakikası + personel maliyeti
```

Çünkü dakika ile para birimi doğrudan toplanamaz.

Bu notebook'ta bekleme süresi önce parasal karşılığa çevrilir:

\[
C_{wait}
=
(\text{toplam bekleme dakikası})
\times
(\text{bekleme dakikası başına maliyet})
\]

Toplam amaç:

\[
C_{total}
=
C_{staff}
+
C_{wait}
+
C_{service\ penalty}
\]

olarak tanımlanır.

In [ ]:
from pathlib import Path
import sys
import warnings
from itertools import product

import numpy as np
import matplotlib.pyplot as plt

from sklearn.exceptions import ConvergenceWarning

PROJE_KOKU = Path.cwd()
if not (PROJE_KOKU / "src").exists() and (PROJE_KOKU.parent / "src").exists():
    PROJE_KOKU = PROJE_KOKU.parent

if not (PROJE_KOKU / "src").exists():
    raise FileNotFoundError(
        "Bu notebook'u repository kök dizininden çalıştırın."
    )

sys.path.insert(0, str((PROJE_KOKU / "src").resolve()))

from discrete_bo import AyrikGaussianProcessBayesOptimizer
from uretim_simulasyonu import politika_degerlendir

warnings.filterwarnings("ignore", category=ConvergenceWarning)

## 3. Simülasyon varsayımları

Temel eğitim modeli:

- 3 vardiya
- vardiya süresi: 240 dakika
- dakika bazında Poisson gelişler
- vardiyalara göre geliş hızları: `1.0`, `1.6`, `1.2` müşteri/dakika
- hizmet süresi yaklaşık normal dağılımlı
- her vardiya kendi kuyruğuna sahip
- vardiyalar arasında kuyruk devri yok
- vardiya başlangıç saatleri sabit

Bu son madde önemlidir: önceki kodda `shift_starts` değişkeni tanımlanmış olmasına rağmen gerçekten optimize edilmiyordu. Bu notebook'ta böyle bir iddia yoktur.

Vardiya başlangıç saatlerini optimize etmek için zaman boyunca değişen aktif personel sayısını ve vardiya örtüşmelerini temsil eden ayrı bir model gerekir.

## 4. Common Random Numbers

Stokastik simülasyonda iki personel politikasını karşılaştırırken farklı rastgele senaryolar kullanmak karşılaştırma gürültüsünü artırabilir.

Bu notebook'ta tüm aday politikalar aynı replikasyon seed'leri ile değerlendirilir:

```python
tekrar_seedleri = [100, 101, ..., 109]
```

Bu, **common random numbers** yaklaşımının basit bir uygulamasıdır.

Amaç, örneğin `[7, 10, 7]` ile `[8, 10, 7]` politikalarını mümkün olduğunca aynı talep ve hizmet koşullarında karşılaştırmaktır.

In [ ]:
tekrar_seedleri = list(range(100, 110))

def amac_fonksiyonu(personel):
    personel = np.asarray(personel, dtype=int)

    ozet = politika_degerlendir(
        personel=personel,
        tekrar_seedleri=tekrar_seedleri,
        saatlik_personel_maliyeti=120.0,
        bekleme_dakika_maliyeti=2.0,
        hedef_vardiya_bekleme_dakika=1.5,
        ceza_katsayisi=1500.0,
    )

    return ozet.toplam_maliyet

## 5. Aday çözüm uzayı

Her vardiyada 5 ile 10 arasında personel kullanılabileceğini varsayalım.

Toplam aday sayısı:

\[
6^3 = 216
\]

Bu küçük eğitim probleminde exhaustive search mümkündür. Ancak gerçek problemde:

- daha fazla vardiya,
- daha geniş personel aralığı,
- ek kapasite kararları,
- pahalı simülasyon replikasyonları

olduğunda her adayı değerlendirmek maliyetli hale gelebilir.

Küçük örneğin avantajı, Bayes optimizasyonunun bulduğu çözümü tam taramayla doğrulayabilmemizdir.

In [ ]:
adaylar = np.array(
    list(product(range(5, 11), repeat=3)),
    dtype=float,
)

print("Toplam aday sayısı:", len(adaylar))

## 6. Ayrık Gaussian Process Bayes optimizasyonu

Burada sürekli acquisition optimization yapmak yerine, GP tüm henüz denenmemiş tamsayı adaylar üzerinde Expected Improvement hesaplar.

Sıradaki simülasyon koşumu en yüksek EI değerine sahip denenmemiş adayda yapılır.

Bu yaklaşım sonlu ve yönetilebilir aday kümelerinde oldukça anlaşılırdır.

In [ ]:
optimizer = AyrikGaussianProcessBayesOptimizer(
    amac_fonksiyonu=amac_fonksiyonu,
    aday_noktalar=adaylar,
    baslangic_noktasi_sayisi=10,
    xi=0.01,
    random_state=1,
)

sonuc = optimizer.optimize_et(
    iterasyon_sayisi=40,
    ayrintili=True,
)

en_iyi_personel = sonuc.en_iyi_x.astype(int)

print("Bayes optimizasyonunun bulduğu personel planı:", en_iyi_personel)
print("Bulunan toplam maliyet:", sonuc.en_iyi_y)
print("Toplam simülasyon değerlendirmesi:", len(sonuc.y_gozlenen))

## 7. Bulunan çözümün operasyonel özeti

In [ ]:
ozet = politika_degerlendir(
    personel=en_iyi_personel,
    tekrar_seedleri=tekrar_seedleri,
    saatlik_personel_maliyeti=120.0,
    bekleme_dakika_maliyeti=2.0,
    hedef_vardiya_bekleme_dakika=1.5,
    ceza_katsayisi=1500.0,
)

print("Toplam maliyet:", round(ozet.toplam_maliyet, 2))
print("Personel maliyeti:", round(ozet.personel_maliyeti, 2))
print("Bekleme maliyeti:", round(ozet.bekleme_maliyeti, 2))
print("Hizmet seviyesi cezası:", round(ozet.hizmet_seviyesi_cezasi, 2))
print("Ortalama bekleme (dk/müşteri):", round(ozet.ortalama_bekleme_dakika, 3))
print(
    "Vardiya bazlı ortalama bekleme:",
    np.round(ozet.vardiya_bazli_ortalama_bekleme, 3),
)
print("Ortalama günlük müşteri sayısı:", round(ozet.ortalama_musteri_sayisi, 1))

## 8. Yakınsama grafiği

In [ ]:
kumulatif_en_iyi = np.minimum.accumulate(sonuc.y_gozlenen)

plt.figure(figsize=(9, 5))
plt.plot(
    np.arange(1, len(kumulatif_en_iyi) + 1),
    kumulatif_en_iyi,
    marker="o",
)
plt.xlabel("Simülasyon değerlendirme sayısı")
plt.ylabel("O ana kadarki en iyi toplam maliyet")
plt.title("Personel optimizasyonu yakınsama grafiği")
plt.grid(True)
plt.show()

## 9. Exhaustive search ile doğrulama

Bu örnekte yalnızca 216 aday olduğu için tüm adayları tarayabiliriz.

Gerçek pahalı problemlerde bunu yapmak istemeyebiliriz; fakat eğitim probleminde BO'nun performansını kontrol etmek için faydalıdır.

In [ ]:
tam_arama_sonuclari = []

for aday in adaylar:
    maliyet = amac_fonksiyonu(aday)
    tam_arama_sonuclari.append((maliyet, aday.astype(int)))

tam_arama_sonuclari.sort(key=lambda kayit: kayit[0])

tam_en_iyi_maliyet, tam_en_iyi_personel = tam_arama_sonuclari[0]

print("Tam arama optimum personel planı:", tam_en_iyi_personel)
print("Tam arama minimum maliyeti:", tam_en_iyi_maliyet)
print(
    "Bayes optimizasyonu ile fark:",
    sonuc.en_iyi_y - tam_en_iyi_maliyet,
)

## 10. Sonucun doğru yorumlanması

Bu küçük problemde Bayes optimizasyonu tam aramayla aynı çözümü bulursa güzel bir doğrulama elde ederiz.

Fakat genel durumda doğru ifade:

```text
Bayes optimizasyonu değerlendirme bütçesi altında en iyi bulunan çözümü verir.
```

olmalıdır.

"Global optimum kesin bulundu" ifadesi, yalnızca problem ayrıca tam yöntemle doğrulanmışsa kullanılmalıdır.

## 11. Modelin sınırlamaları

Bu örnek özellikle öğretim için basitleştirilmiştir.

Gerçek personel planlamasında eklenebilecek unsurlar:

- vardiya başlangıç/bitiş saatleri
- vardiya örtüşmeleri
- mola planları
- beceri seviyeleri
- farklı müşteri sınıfları
- öncelikli kuyruklar
- devamsızlık
- overtime
- servis seviyesi yüzdeleri
- zaman dilimine göre değişen arrival rate
- vardiyalar arası kuyruk devri

Bu durumda karar uzayı ve simülasyon modeli daha gerçekçi hale gelir.